# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install -q mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

print("Dataset Title: ", dataset.metadata.name)
print("Description: ", dataset.metadata.description)
print("Version: ", dataset.metadata.version)
print("Published: ", dataset.metadata.datePublished)
print("Keywords: ", dataset.metadata.keywords)
print("License: ", dataset.metadata.license)
print("Croissant schema conforms to: ", dataset.metadata.conformsTo)

## 2. Data Overview
Review available record sets and their fields (by `@id`).

In [ ]:
# List all available record sets, their @id, and their fields' @id
record_sets = dataset.metadata.recordSet
if not record_sets:
    print("No record sets found in the dataset metadata.")
else:
    for recset in record_sets:
        print(f"RecordSet name: {recset.name}")
        print(f"  @id: {recset['@id']}")
        print(f"  Description: {recset.description if hasattr(recset, 'description') else ''}")
        if hasattr(recset, 'field'):
            print("  Fields:")
            for f in recset.field:
                print(f"    - name: {f.name}, @id: {f['@id']}, dataType: {f.dataType if hasattr(f, 'dataType') else ''}")
        print()

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis, referencing the record set and field `@id`s identified above.

In [ ]:
# If record sets are found, extract their @id
available_record_sets = []
if dataset.metadata.recordSet:
    # Gather all record set @id values
    for rset in dataset.metadata.recordSet:
        available_record_sets.append(rset['@id'])
    print(f"Record sets found: {available_record_sets}")
else:
    print("No record sets found. Please check the Croissant schema.")

# Example usage with the first record set
if available_record_sets:
    dataframes = {}
    for record_set_id in available_record_sets:
        # Load all records from the record set
        print(f"Loading records from RecordSet @id: {record_set_id}")
        records = list(dataset.records(record_set=record_set_id))
        if records:
            df = pd.DataFrame(records)
            dataframes[record_set_id] = df
            print(f"Loaded DataFrame for RecordSet {record_set_id} with columns:")
            print(df.columns.tolist())
            display(df.head(3))
        else:
            print(f"No records found for RecordSet @id: {record_set_id}")
else:
    dataframes = {}

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps: filtering records based on criteria, normalizing numeric fields, and grouping data by key attributes. All data elements referenced by their `@id`.

In [ ]:
# Please update these IDs based on output from Section 2/3 if you know specific ones
import numpy as np

if dataframes:
    # Select the first available DataFrame
    example_record_set_id = list(dataframes.keys())[0]
    df = dataframes[example_record_set_id]

    # Pick a first numeric column (by checking dtypes)
    numeric_candidates = df.select_dtypes(include=[np.number]).columns.tolist()
    if numeric_candidates:
        numeric_field = numeric_candidates[0]  # Use the column name as the field @id
        print(f"Using numeric field '@id': {numeric_field}")
        threshold = df[numeric_field].mean()
        filtered_df = df[df[numeric_field] > threshold]

        print(f"Filtered records with {numeric_field} > {threshold:.2f}:")
        print(filtered_df.head())

        filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"Normalized {numeric_field} for filtered records:")
        print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Try grouping by a textual field (categorical)
        group_candidates = df.select_dtypes(include=[object]).columns.tolist()
        group_field = None
        for g in group_candidates:
            if df[g].nunique() < len(df) // 2:
                group_field = g
                break
        if group_field:
            grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
            print(f"Grouped data by {group_field}:")
            print(grouped_df.head())
        else:
            print("No suitable textual field found for grouping.")
    else:
        print("No numeric fields available for EDA.")
else:
    print("No data available for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset. (Fields are referenced by `@id`/column name.)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if dataframes:
    df = list(dataframes.values())[0]
    # Again use the same numeric field as in EDA
    numeric_candidates = df.select_dtypes(include=[np.number]).columns.tolist()
    if numeric_candidates:
        num_field = numeric_candidates[0]
        plt.figure(figsize=(8, 4))
        sns.histplot(df[num_field].dropna(), kde=True, bins=30)
        plt.title(f'Distribution of `{num_field}`')
        plt.xlabel(num_field)
        plt.show()

        # If a group field was identified for grouping, show a boxplot
        group_candidates = df.select_dtypes(include=[object]).columns.tolist()
        if group_candidates:
            group_field = group_candidates[0]
            plt.figure(figsize=(10, 5))
            sns.boxplot(x=group_field, y=num_field, data=df)
            plt.title(f'{num_field} grouped by {group_field}')
            plt.show()
else:
    print("Nothing to visualize; no dataframes.")

## 6. Conclusion
In this notebook, we demonstrated how to load, inspect, and process a FAIR-compliant dataset using the `mlcroissant` library. We referenced all entities using their `@id`, extracted records for analysis, applied basic EDA and normalization, and visualized key numeric fields. For deeper inspection, consult the Croissant schema, the `mlcroissant` documentation, or the original dataset publication for precise field meanings.

You can modify this notebook to reference other record sets and fields using their `@id` for domain-specific analysis.